# Explicit Backfill Operator Tutorial: Fabric Lakehouse

Continue after step 9 of the [Manifest Package Operator Tutorial](manifest_package_operator_tutorial.ipynb). Import this notebook into **Microsoft Fabric**, attach your deployed default Lakehouse and matching published Environment, and map the package, original video tree, and model folders to Lakehouse paths.

This notebook hashes your **actual model bytes**, generates the backfill catalog, previews the package, and triggers the deployed Control SJD after explicit confirmation. No fabricated model hashes, repository checkout, Azure CLI login, or manual registration JSON is required.

> **Safe default:** `RUN_SUBMISSION = False`. Catalog generation and preview read local/mounted files and write the catalog only. Publication and registration require enabling submission and setting the target confirmation explicitly. This notebook does not deploy infrastructure, approve models, enable schedules, or perform inference in the notebook session.


## 1. Before you start

1. Import this `.ipynb` into Fabric as a **PySpark notebook**. Attach the same default Lakehouse used by the production SJDs and a published Environment containing the matching `people-counter` SDK release, `azure-identity`, and `azure-storage-file-datalake`. The SDK must include this notebook's `--fabric` support. Publish the compatible SDK/Environment through the normal release process before submitting; do not install a different SDK over a running kernel.
2. If the manifest notebook ran with `FABRIC_MODE = True`, its default package and source-video shortcut paths already match this notebook; use the same Lakehouse and no package copying is needed. If preparation ran locally, copy or map the **complete manifest package** and unchanged **source video tree** into the Lakehouse. Also provide the approved **model tree**. Use Python-accessible mounted paths such as `/lakehouse/default/Files/...`, not `abfss://` strings with `Path`. If you customized `PACKAGE_DIR` or `VIDEO_ROOT` in the first notebook, use those same paths here. Relative video paths must remain unchanged: if preparation recorded `samples/clip.mp4`, this notebook's video root must contain `samples/clip.mp4`.
3. Obtain the actual Environment and Control SJD IDs and source ADLS account/filesystem from your deployment administrator. Workspace and Lakehouse IDs are read from the attached default Lakehouse context. The source shortcut (for example `Files/videos`) must expose the ADLS filesystem root; the deployed workers still read its published `incoming/...` paths.
4. Use **explicit intake**. Disable or coordinate competing arrival-driven intake. Ensure Dispatcher, Reconciliation, Gold, and Refresh are configured. This notebook cannot detect an independent arrival rule or safely register the same processing run twice.
5. The notebook execution identity needs ADLS publication access, OneLake model/release/shortcut read and backfill-intake write access, and Control SJD invoke/read permission. The Fabric runtime identity must also read the published videos and models. Pipeline/service-principal permissions can differ from interactive user permissions; token/API failures stop the run, with no CLI fallback.

Authentication uses short-lived `notebookutils` storage and Fabric API tokens. Tokens are never printed or written to the catalog/state. Optional processing monitoring uses the active Spark session, not a local DuckDB/Azure CLI credential chain.

See the [explicit backfill guide](../docs/explicit-backfill.md) for deployment setup and the unchanged local CLI workflow.


## 2. Choose Fabric or local defaults and configure deployment bindings

Set `FABRIC_MODE` first. The `if` branch uses mounted Lakehouse paths; the `else` branch uses repository-local paths matching the manifest tutorial, including `offline-models` as the model root. Edit paths, target, and profile inside the branch you use. Switching modes and rerunning this cell replaces all path defaults together; it does not create, copy, or move your inputs. Set real Environment/Control IDs and ADLS account/filesystem in `TARGET_BINDING` below. Set `table_directory` to `Tables/dbo` for the default schema, `Tables/<schema>` for another schema, or `Tables` for a non-schema Lakehouse.

`MODELS_DIR` is the parent of the SDK's `rtdetr_osnet/...` model hierarchy. All four artifacts for the selected preset must exist, including the detector configuration and preprocessor files. These must be the same reviewed bytes deployed under the target Lakehouse's `Files/models/`; hashing does not approve or upload them. A different mapped model folder is allowed, but submission checks its hashes against the deployed bytes.

Keep `CATALOG_PATH` and `STATE_DIR` outside the immutable package. Retain the state directory on the Lakehouse for every retry; never use an ephemeral `/tmp` directory for registration evidence or run two notebook/CLI instances against the same state. Fabric publication uses a temporary local SQLite cache (not SQLite on OneLake); retries reverify remote publications while retaining durable registration/job evidence.

For local testing, keep `FABRIC_MODE = False` and run the notebook from the repository root or a folder inside it (such as `notebooks/`). The local branch finds the repository root and supplies local path defaults, but you must still supply real workspace/Lakehouse IDs and deployment bindings before generating a catalog. Set `FABRIC_MODE = True` after importing into Fabric. Local submission keeps the CLI's existing Azure credentials and optional DuckDB monitor.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import json
import sys

from people_counter import backfill_operator as operator

FABRIC_MODE = False

if FABRIC_MODE:
    LAKEHOUSE_ROOT = Path("/lakehouse/default")
    PACKAGE_DIR = LAKEHOUSE_ROOT / "Files/backfill/manifest-package-tutorial"
    VIDEO_ROOT = LAKEHOUSE_ROOT / "Files/backfill/source-videos"
    MODELS_DIR = LAKEHOUSE_ROOT / "Files/models"
    CATALOG_PATH = LAKEHOUSE_ROOT / "Files/backfill/config/backfill-catalog.json"
    STATE_DIR = LAKEHOUSE_ROOT / "Files/backfill/state/manifest-package-tutorial"
    TARGET = "production"
    PROFILE = "rtdetr-r18-onnx-cpu"

    import notebookutils
    context = notebookutils.runtime.context
    workspace_id = context["defaultLakehouseWorkspaceId"]
    lakehouse_id = context["defaultLakehouseId"]
else:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "samples").is_dir():
            REPO_ROOT = candidate
            break
    else:
        raise RuntimeError("Run local mode from inside the people-counter repository")

    PACKAGE_DIR = REPO_ROOT / "samples/manifest-package-tutorial"
    VIDEO_ROOT = REPO_ROOT
    MODELS_DIR = REPO_ROOT / "offline-models"
    CATALOG_PATH = REPO_ROOT / "config/backfill-catalog.json"
    STATE_DIR = REPO_ROOT / "state/manifest-package-tutorial-backfill"
    TARGET = "production"
    PROFILE = "rtdetr-r18-onnx-cpu"

    workspace_id = "<workspace-uuid>"
    lakehouse_id = "<lakehouse-uuid>"

TARGET_BINDING = {
    "workspace_id": workspace_id,
    "lakehouse_id": lakehouse_id,
    "environment_id": "<published-environment-uuid>",
    "control_sjd_id": "<control-sjd-uuid>",
    "intake_mode": "explicit",
    "storage_account": "<source-storage-account>",
    "filesystem": "<source-filesystem>",
    "shortcut_path": "Files/videos",
    "table_directory": "Tables/dbo",
}


def run_backfill(arguments: list[str]) -> dict:
    output = StringIO()
    with redirect_stdout(output), operator.progress_reporter(
        lambda message: print(message, file=sys.stderr, flush=True)
    ):
        exit_code = operator.main(arguments)
    if exit_code != 0:
        raise RuntimeError(
            f"Backfill command exited with code {exit_code}: {output.getvalue().strip()}. "
            "Retain the state directory. Accepted Fabric jobs are not cancelled; see step 8."
        )
    return json.loads(output.getvalue())


print(f"Package: {PACKAGE_DIR}\nVideo root: {VIDEO_ROOT}\nModels: {MODELS_DIR}")
print(f"Catalog output: {CATALOG_PATH}\nRetained state: {STATE_DIR}")
print(f"Target workspace: {workspace_id}; Lakehouse: {lakehouse_id}")


## 3. Generate the catalog from your model files

This cell calls the SDK's existing `prepare_catalog` function. It streams each required model artifact from `MODELS_DIR`, calculates its real SHA-256, expands effective profile settings, and derives profile/runtime identities. **There are no all-zero placeholder hashes.** Missing files and invalid deployment bindings fail before the catalog is written. No models are downloaded, exported, approved, or uploaded.

Rerunning with the same model bytes/binding reuses the same catalog. Changed bytes or bindings cannot overwrite an existing target/profile definition: review the revision and choose a new catalog and coordinated processing run, rather than deleting retry state. Do not reuse the previous preview-only sample catalog; use the new path configured above.

Generating this file does not establish cloud readiness. Review its binding and the hashes below before proceeding.


In [ ]:
if CATALOG_PATH.resolve().is_relative_to(PACKAGE_DIR.resolve()):
    raise ValueError("CATALOG_PATH must remain outside the immutable manifest package")

catalog_report = operator.prepare_catalog(
    TARGET, TARGET_BINDING, MODELS_DIR, [PROFILE], CATALOG_PATH,
)
resolved = operator.load_target(CATALOG_PATH, TARGET, PROFILE)
print(json.dumps(catalog_report, indent=2))
print("SHA-256 from the exact model files:")
for relative_path, digest in resolved.profile["model_artifact_sha256"].items():
    print(f"{digest}  {MODELS_DIR / relative_path}")


In [ ]:
operator.main(["--catalog", str(CATALOG_PATH), "--list-targets"])


In [ ]:
operator.main([
    "--catalog", str(CATALOG_PATH), "--target", TARGET, "--list-profiles",
])


In [ ]:
print(json.dumps(operator.load_target(CATALOG_PATH, TARGET, PROFILE).profile, indent=2))


## 4. Preview the completed manifest package

This reads the package and rehashes every referenced source video through the mounted path. It does not construct cloud clients, register work, or write submission state. Large video sets can take time. If files were transferred, `VIDEO_ROOT` must preserve the relative tree recorded by the first notebook.

Review target IDs, ADLS destination, video/partition counts, package digest, and the complete effective profile with the **real model artifact SHA-256 values**. Cloud release/Environment/model/shortcut checks happen again on submission.


In [ ]:
backfill_args = [
    "--manifest-package-dir", str(PACKAGE_DIR.resolve()),
    "--video-root", str(VIDEO_ROOT.resolve()),
    "--catalog", str(CATALOG_PATH.resolve()),
    "--target", TARGET,
    "--profile", PROFILE,
    "--state-dir", str(STATE_DIR.resolve()),
]
if FABRIC_MODE:
    backfill_args.append("--fabric")
preview = run_backfill(backfill_args)
reviewed_args = backfill_args.copy()
reviewed_target = TARGET
print(f"Target: {reviewed_target}; profile: {preview['profile_id']}")
print(f"Videos: {preview['item_count']}; registration partitions: {preview['partition_count']}")
print(json.dumps(preview, indent=2))


## 5. Review and enable submission

Confirm the administrator-reviewed target, explicit intake, matching deployed model bytes and SDK release, correct videos/profile, and exclusive use of the retained state directory. Hash generation is not model approval.

Leave `RUN_SUBMISSION = False` to stay at preview. After review, set it to `True` and set `SUBMISSION_CONFIRMATION = "SUBMIT production"` (replace `production` if you selected another target). Fabric uses this explicit cell value, not an interactive stdin prompt. `WAIT_FOR_PROCESSING = True` monitors exact registered work IDs through the active Fabric Spark session until all have committed successful attempts. This access is preflighted before publication. Set it to `False` to stop after Control registration.

Processing monitoring has **no timeout**: paused Dispatcher schedules or unavailable capacity can leave work pending. Progress appears about every 15 seconds. Neither registration nor successful processing proves that Gold or report refresh is complete.


In [ ]:
RUN_SUBMISSION = False
SUBMISSION_CONFIRMATION = ""
WAIT_FOR_PROCESSING = True


## 6. Publish and trigger backfill

The cell revalidates the offline preview and requires the exact `SUBMIT <target>` confirmation set in section 5. It then uses the current Fabric notebook identity to verify release evidence, Environment binding, and deployed model hashes; publish videos before final manifests; verify the source shortcut; write immutable registration partitions; and invoke the deployed **Control SJD**. It retains and polls job locations, rather than submitting duplicates on retry.

Notebook and Control target must agree with the **attached default Lakehouse**. Configure the correct attachment before starting, including when a notebook lives in a different workspace. This workflow still publishes to the configured ADLS source and registers its verified shortcut paths; it is not an unvalidated direct-ingest shortcut.

Use **Interrupt execution** to leave a long wait. An interrupt stops the local wait, not accepted Fabric jobs; some partitions may still be unsubmitted. Preserve state and follow step 8. Explicit operator confirmation is intentional; do not schedule this tutorial with submission enabled.


In [ ]:
submission_report = None
if not RUN_SUBMISSION:
    print("Submission disabled. Nothing was uploaded or registered by this cell.")
else:
    if backfill_args != reviewed_args or run_backfill(backfill_args) != preview:
        raise RuntimeError("The preview changed. Rerun step 4 and review it before submission.")
    expected_confirmation = f"SUBMIT {reviewed_target}"
    confirmation = SUBMISSION_CONFIRMATION if FABRIC_MODE else input(
        f"Cloud writes enabled. Type {expected_confirmation!r} to continue: "
    )
    if confirmation != expected_confirmation:
        raise ValueError("Submission confirmation did not match. No cloud writes were started.")
    submit_flags = ["--submit", "--wait"] if WAIT_FOR_PROCESSING else ["--submit"]
    submission_report = run_backfill([*reviewed_args, *submit_flags])
    print(json.dumps(submission_report, indent=2))


## 7. Read the result and retain evidence

| Result | What it proves | Next action |
|---|---|---|
| `preview` | Package/video validation only | Review and approve submission |
| `registered` | Control registration jobs completed | Monitor exact work IDs for committed processing |
| `processed` with `processing_complete: true` | Every submitted work ID has a successful committed attempt | Verify Gold and reporting in step 9 |
| Error or `monitoring_stopped` (exit 130) | No completion guarantee; remote jobs can continue | Retain state and recover in step 8 |

Keep the entire Lakehouse state directory: publication reports, `registration-plan.json`, exact partitions, job-state files/locations, `publication-evidence.json`, and summaries. The registration plan contains the exact work IDs to monitor. Fabric's temporary local publication cache is intentionally disposable; the verified remote objects and durable evidence support retries.

This read-only cell displays retained summaries. A summary from an earlier attempt is evidence, not proof that the latest cell completed.


In [ ]:
retained_state = Path(reviewed_args[reviewed_args.index("--state-dir") + 1])
print(f"Keep this directory for retries: {retained_state}")
for name in ("registration-summary.json", "processing-summary.json"):
    path = retained_state / name
    if path.is_file():
        print(f"\nRetained evidence: {path}")
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2))
    else:
        print(f"Not yet available: {name}")


## 8. Resume safely

Keep the same package/video tree, real model bytes, catalog/target/profile, SDK release, and persistent state directory. After repairing the cause, rerun setup, catalog generation, and preview, review again, and explicitly submit with the same state. Existing publications are reverified and retained Control jobs are polled instead of duplicated. Never delete state to bypass a conflict or run two operators concurrently.

| Problem | Safe next action |
|---|---|
| Missing model artifact | Set `MODELS_DIR` to the correct hierarchy or restore the complete approved model set; no dummy hashes are substituted |
| Placeholder IDs / no default Lakehouse | Fill in real deployment bindings and attach the target Lakehouse before generating the catalog |
| Different existing catalog definition | Review the change with the administrator; use a new catalog/run only for an intentional revision |
| Package/video checksum or missing-video error | Restore unchanged inputs or correct preparation in the first notebook; never patch package JSON |
| Release, Environment, model, shortcut, or authorization mismatch | Repair the target or supply matching SDK/model bytes/binding; no CLI credential fallback |
| State belongs to a different package/target/profile/release | Recover the original run; do not bypass the conflict with an empty state directory |
| `submission outcome unknown` | Stop and ask an administrator to inspect Fabric history and recover the matching job location using the [recovery guide](../docs/explicit-backfill.md#4-retry-and-recovery) |
| Failed/cancelled/deduplicated/timed-out Control job | Investigate retained job state before permitting any replacement submission |
| Processing remains pending | Check Dispatcher and capacity; there is no processing timeout |
| Failed/DEAD/quarantined/cancelled work or query/access error | Inspect the exact work ID, status, and last error; monitoring stops rather than claiming success |

Changing the profile, model bytes, or SDK/release is a new processing run, not recovery. Do not manually claim batches.


## 9. Verify processing and reporting separately

Use the [production runbook](fabric/README.md) with your administrator:

- Require `status == SUCCEEDED` and a nonempty `committed_attempt_id` in `people_counter_sjd_work` for every exact `work_id` in `registration-plan.json`. Wait mode checks this automatically; registration-only mode does not.
- Require the matching `(work_id, attempt_id)` in `people_counter_sjd_gold_video`. Investigate missing IDs, duplicates, retries, `DEAD` work, and `last_error`.
- Require successful Reconciliation, Gold, and Refresh runs, no unresolved reconciliation errors, and an empty semantic-refresh outbox before declaring reporting complete.

A completed Spark job alone is not proof of committed processing or refreshed reports. This notebook triggers and monitors backfill; it does not certify the separate reporting checks.
